# Correlation analysis

In [ ]:
import pandas as pd
import glob
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

In [18]:
# Load all CSVs
files = glob.glob("test_csvs/*.csv")

In [19]:
for f in files:
    df = pd.read_csv(f)
    corr = df.corr(numeric_only=True)
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
    plt.title(Path(f).stem)
    plt.tight_layout()
    plt.savefig(f"corr_plots/test_corr_{Path(f).stem}.png", dpi=150)
    plt.close()
    # Check for high correlations
    print(f"\n=== {Path(f).stem} ===")
    found = False
    for col in corr.columns:
        for row in corr.index:
            if col != row and corr.loc[row, col] > 0.5:  # abs check below
                if abs(corr.loc[row, col]) > 0.5:
                    print(f"  {row} <-> {col}: {corr.loc[row, col]:.2f}")
                    found = True
    if not found:
        print("  No correlations above 0.5 found")


=== ml_ready_data_wavelet_method_db2 ===
  max <-> mean: 0.86
  min <-> mean: 0.59
  mean <-> max: 0.86
  mean <-> min: 0.59
  wavelet_level_4_energy_ratio <-> wavelet_level_3_energy_ratio: 0.50
  wavelet_level_5_energy_ratio <-> wavelet_level_3_energy_ratio: 0.52
  wavelet_level_1_entropy <-> wavelet_level_0_entropy: 0.85
  wavelet_level_2_entropy <-> wavelet_level_0_entropy: 0.91
  wavelet_level_3_entropy <-> wavelet_level_0_entropy: 0.94
  wavelet_level_4_entropy <-> wavelet_level_0_entropy: 0.96
  wavelet_level_5_entropy <-> wavelet_level_0_entropy: 0.96
  wavelet_level_0_entropy <-> wavelet_level_1_entropy: 0.85
  wavelet_level_2_entropy <-> wavelet_level_1_entropy: 0.78
  wavelet_level_3_entropy <-> wavelet_level_1_entropy: 0.80
  wavelet_level_4_entropy <-> wavelet_level_1_entropy: 0.81
  wavelet_level_5_entropy <-> wavelet_level_1_entropy: 0.78
  wavelet_level_0_entropy <-> wavelet_level_2_entropy: 0.91
  wavelet_level_1_entropy <-> wavelet_level_2_entropy: 0.78
  wavelet_leve

In [21]:
from scipy import stats
for f in files:
    df = pd.read_csv(f)

    feature_cols = [c for c in df.columns if c not in ['file', 'scan_number', 'in_control']]

    results = []
    for col in feature_cols:
        corr, pval = stats.pointbiserialr(df['in_control'], df[col])
        results.append({'feature': col, 'correlation': corr, 'pval': pval})

    results_df = pd.DataFrame(results).sort_values('correlation', key=abs, ascending=False)
    print(results_df)

                         feature  correlation           pval
1                           mean    -0.284079   0.000000e+00
2                            max    -0.246546   0.000000e+00
3                            min    -0.184275  3.480615e-268
5   wavelet_level_0_energy_ratio    -0.109825   1.640499e-95
9        wavelet_level_0_entropy    -0.107184   4.866543e-91
4                         stddev    -0.105252   7.748045e-88
0             mean_laser_current    -0.018713   4.268943e-04
6   wavelet_level_1_energy_ratio          NaN            NaN
7   wavelet_level_2_energy_ratio          NaN            NaN
8   wavelet_level_3_energy_ratio          NaN            NaN
10       wavelet_level_1_entropy          NaN            NaN
11       wavelet_level_2_entropy          NaN            NaN
12       wavelet_level_3_entropy          NaN            NaN
13  wavelet_level_4_energy_ratio          NaN            NaN
14  wavelet_level_5_energy_ratio          NaN            NaN
15       wavelet_level_4

## Empty columns

In [ ]:
#from sklearn.preprocessing import StandardScaler
#from sklearn.decomposition import PCA
df = pd.read_csv("test_csvs/test_data_wavelet_method_db4.csv")
wavelet_cols = [c for c in df.columns if 'wavelet' in c]
print(df[wavelet_cols].isna().sum())
"""
X = StandardScaler().fit_transform(df[wavelet_cols])

pca = PCA(n_components=0.95)  # keep enough components to explain 95% variance
X_pca = pca.fit_transform(X)

pca_cols = [f'wavelet_pca_{i}' for i in range(X_pca.shape[1])]
df_pca = pd.DataFrame(X_pca, columns=pca_cols)
"""

wavelet_level_0_energy_ratio      0
wavelet_level_1_energy_ratio    154
wavelet_level_2_energy_ratio    363
wavelet_level_0_entropy           0
wavelet_level_1_entropy         154
wavelet_level_2_entropy         363
dtype: int64


"\nX = StandardScaler().fit_transform(df[wavelet_cols])\n\npca = PCA(n_components=0.95)  # keep enough components to explain 95% variance\nX_pca = pca.fit_transform(X)\n\npca_cols = [f'wavelet_pca_{i}' for i in range(X_pca.shape[1])]\ndf_pca = pd.DataFrame(X_pca, columns=pca_cols)\n"

In [36]:
df1 = pd.read_csv("ml_ready_csvs/ml_ready_data_wavelet_method_db3.csv")
wavelet_cols = [c for c in df1.columns if 'wavelet' in c]
print(df1[wavelet_cols].isna().sum())

wavelet_level_0_energy_ratio       0
wavelet_level_1_energy_ratio     154
wavelet_level_2_energy_ratio     363
wavelet_level_0_entropy            0
wavelet_level_1_entropy          154
wavelet_level_2_entropy          363
wavelet_level_3_energy_ratio     731
wavelet_level_4_energy_ratio    5783
wavelet_level_5_energy_ratio    8458
wavelet_level_3_entropy          731
wavelet_level_4_entropy         5783
wavelet_level_5_entropy         8458
dtype: int64


## Try to recreate the wavelets, set as test_csvs folder

In [10]:
import os

import numpy as np
import pandas as pd
import scipy.stats as stats
from tqdm import tqdm
import pywt

In [11]:
def simple_statistics(time_series):
    # time_series is a numpy array
    # return the mean, max, min, stddev
    return {
        'mean': np.mean(time_series),
        'max': np.max(time_series),
        'min': np.min(time_series),
        'stddev': np.std(time_series)
    }

# TODO: tune this so it's actually good
# 1. could add entropy of c**2 as an additional feature to capture "spikiness" so we don't just
# average over time
# 2. try db2, db3, db4, db6, db8, sym3, sym6, coif3.

def wavelet_features(signal, wavelet='db4', level=5, include_entropy=False):
    # db4 means that there's 4 vanishing moments, only picks up on oscillations beyond cubic
    coeffs = pywt.wavedec(signal, wavelet, level=min(level, pywt.dwt_max_level(len(signal), wavelet)))
    # Energy in each sub-band (approximation + details)
    energies = [np.sum(c**2) / len(c) for c in coeffs]
    total = sum(energies)
    # Issue with nan/division by 0 in some files
    #if total == 0:
    #    return {f'wavelet_level_{i}_energy_ratio': 0 for i in range(len(coeffs))}
    result = {f'wavelet_level_{i}_energy_ratio': e/total for i, e in enumerate(energies)}
    if include_entropy:
        for i, c in enumerate(coeffs):
            result[f'wavelet_level_{i}_entropy'] = stats.entropy(c**2)
    return result


In [17]:
data_dir = "./csvs3/"
new_data_dir = "./test_csvs/"
os.makedirs(new_data_dir, exist_ok=True)

files = os.listdir(data_dir)
files = [f for f in files if f.endswith(".csv")]
files.sort(key=lambda x: int(x.split(".")[0][5:]))
do_this_many = 379 # len(files) # NOTE: can be less if you want to test

for wavelet_method in [ 'db4', 'sym3']:
    print(f"Starting {wavelet_method}...")
    # parse all files and prepare to put them in a single DataFrame
    df_list = [] # will contain a list of dictionaries
    for f_idx in tqdm(range(do_this_many)):
        file_path = os.path.join(data_dir, files[f_idx])
        df = pd.read_csv(file_path)

        num_segments = df['scan_number'].nunique()
        for scan_num in range(num_segments):
            segment = df[df['scan_number'] == scan_num]

            if segment.empty:
                continue

            features = {}
            features['file'] = files[f_idx]
            features['scan_number'] = scan_num
            features['mean_laser_current'] = segment['laser_current'].mean()
            features['in_control'] = segment['in_control'].mode().values[0] # most common
            features |= simple_statistics(segment['signal'].values)
            features |= wavelet_features(segment['signal'].values, wavelet=wavelet_method, level=2, include_entropy=True)
            df_list.append(features)
    # create a DataFrame from the list of dictionaries
    combined_df = pd.DataFrame(df_list)
    # save the DataFrame to a new CSV file
    combined_df.to_csv(os.path.join(new_data_dir, f"test_data_wavelet_method_{wavelet_method}.csv"), index=False)

Starting db4...


100%|██████████| 379/379 [01:49<00:00,  3.46it/s]


Starting sym3...


100%|██████████| 379/379 [03:03<00:00,  2.06it/s]
